# Polaris RE — Term Life YRT Pricing: End-to-End Validation

**Purpose:** Demonstrate a complete deal pricing workflow for a YRT reinsurance treaty
on a term life inforce block. Validates that the full pipeline produces actuarially
reasonable results.

**Prerequisites:** Phase 1 implementation complete (all `NotImplementedError` stubs resolved).

**Sections:**
1. Setup and imports
2. Build an inforce block
3. Define assumptions (CIA 2014, duration-based lapse)
4. Project gross term life cash flows
5. Apply YRT treaty
6. Profit test the deal
7. Scenario analysis
8. Validation checks (vs hand calculations)

In [ ]:
# Standard imports
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from datetime import date

from rich import print as rprint

# Polaris RE imports
from polaris_re.core.policy import Policy, ProductType, Sex, SmokerStatus
from polaris_re.core.inforce import InforceBlock
from polaris_re.core.projection import ProjectionConfig
from polaris_re.assumptions.mortality import MortalityTable, MortalityTableSource
from polaris_re.assumptions.lapse import LapseAssumption
from polaris_re.assumptions.assumption_set import AssumptionSet
from polaris_re.products.term_life import TermLife
from polaris_re.reinsurance.yrt import YRTTreaty
from polaris_re.analytics.profit_test import ProfitTester
from polaris_re.analytics.scenario import ScenarioRunner

print('Polaris RE imports OK')

## 1. Build Inforce Block

We use a small representative block for validation. In production this would be
loaded from the synthetic block CSV or a real inforce extract.

In [ ]:
VALUATION_DATE = date(2025, 1, 1)

# Representative term life policies
policies = [
    Policy(
        policy_id='P001', issue_age=35, attained_age=35,
        sex=Sex.MALE, smoker_status=SmokerStatus.NON_SMOKER,
        underwriting_class='PREFERRED',
        face_amount=500_000, annual_premium=700,
        product_type=ProductType.TERM, policy_term=20, duration_inforce=0,
        reinsurance_cession_pct=0.50,
        issue_date=VALUATION_DATE, valuation_date=VALUATION_DATE,
    ),
    Policy(
        policy_id='P002', issue_age=45, attained_age=50,
        sex=Sex.MALE, smoker_status=SmokerStatus.NON_SMOKER,
        underwriting_class='STANDARD',
        face_amount=1_000_000, annual_premium=3_500,
        product_type=ProductType.TERM, policy_term=20, duration_inforce=60,
        reinsurance_cession_pct=0.50,
        issue_date=date(2020, 1, 1), valuation_date=VALUATION_DATE,
    ),
    Policy(
        policy_id='P003', issue_age=40, attained_age=40,
        sex=Sex.FEMALE, smoker_status=SmokerStatus.NON_SMOKER,
        underwriting_class='PREF_PLUS',
        face_amount=750_000, annual_premium=600,
        product_type=ProductType.TERM, policy_term=30, duration_inforce=0,
        reinsurance_cession_pct=0.50,
        issue_date=VALUATION_DATE, valuation_date=VALUATION_DATE,
    ),
]

block = InforceBlock(policies=policies, block_id='VALIDATION_BLOCK_001')

rprint(f'Block: {block.n_policies} policies')
rprint(f'Total face amount: ${block.total_face_amount():,.0f}')
rprint(f'Total annual premium: ${block.total_annual_premium():,.0f}')

## 2. Define Assumptions

In [ ]:
# Load CIA 2014 mortality table
mortality = MortalityTable.load(source=MortalityTableSource.CIA_2014)

# Duration-based lapse rates
lapse = LapseAssumption.from_duration_table({
    1: 0.08, 2: 0.06, 3: 0.05, 4: 0.04, 5: 0.04,
    6: 0.03, 7: 0.03, 8: 0.03, 9: 0.03, 10: 0.03,
    'ultimate': 0.02,
})

assumptions = AssumptionSet(
    mortality=mortality,
    lapse=lapse,
    version='validation-v1.0',
    effective_date=VALUATION_DATE,
    notes='Validation notebook — CIA 2014 + duration lapse',
)

rprint(assumptions.summary)

## 3. Configure and Run Projection

In [ ]:
config = ProjectionConfig(
    valuation_date=VALUATION_DATE,
    projection_horizon_years=30,  # cover the 30-year term policy
    discount_rate=0.05,
    valuation_interest_rate=0.035,
)

# Project gross cash flows
product = TermLife(inforce=block, assumptions=assumptions, config=config)
gross = product.project(seriatim=True)  # seriatim=True for policy-level validation

rprint(f'Projection complete: {gross.projection_months} monthly periods')
rprint(f'Basis: {gross.basis}')

## 4. Validate Gross Cash Flows

Before applying the treaty, verify the gross projection is internally consistent.

In [ ]:
# Accounting identity: net_cash_flow = premiums - claims - expenses - Δreserve
recomputed_ncf = (
    gross.gross_premiums
    - gross.death_claims
    - gross.lapse_surrenders
    - gross.expenses
    - gross.reserve_increase
)
np.testing.assert_allclose(gross.net_cash_flow, recomputed_ncf, rtol=1e-6)
print('✓ Accounting identity holds')

# Reserve must hit 0 at policy expiry
assert gross.reserve_balance[-1] < 1.0, 'Reserve not zero at end of projection'
print('✓ Terminal reserve approximately zero')

# Loss ratio sanity check (should be 40-80% for term life reinsurance)
lr = gross.loss_ratio()
print(f'Loss ratio: {lr:.1%} (expected: 40-80% for well-priced term)')

## 5. Apply YRT Treaty

In [ ]:
treaty = YRTTreaty(
    cession_pct=0.50,
    flat_yrt_rate_per_1000=1.20,  # illustrative: $1.20 per $1,000 NAR per year
    treaty_name='VALIDATION_YRT_001',
)

net, ceded = treaty.apply(gross)

# Verify additivity
treaty.verify_additivity(gross, net, ceded)
print('✓ Treaty additivity verified: net + ceded == gross')

rprint(f'YRT treaty applied. Net basis PV premiums @ 5%: ${net.pv_premiums(0.05):,.0f}')

## 6. Profit Test

In [ ]:
tester = ProfitTester(cashflows=net, hurdle_rate=0.10)
result = tester.run()

print('=' * 50)
print('PROFIT TEST RESULTS')
print('=' * 50)
print(f'IRR:                  {result.irr:.2%}' if result.irr else 'IRR: did not converge')
print(f'PV Profits @ 10%:     ${result.pv_profits:,.0f}')
print(f'Profit Margin:        {result.profit_margin:.2%}')
print(f'Break-even Year:      {result.breakeven_year}')
print(f'Total Undiscounted:   ${result.total_undiscounted_profit:,.0f}')

## 7. Scenario Analysis

In [ ]:
runner = ScenarioRunner(
    inforce=block,
    base_assumptions=assumptions,
    config=config,
    treaty=treaty,
    hurdle_rate=0.10,
)

scenario_result = runner.run()  # uses standard_stress_scenarios() by default

# Display results table
print(f'{'Scenario':<25} {'IRR':>8} {'PV Profit':>12} {'Margin':>8}')
print('-' * 57)
for name, r in scenario_result.scenarios:
    irr_str = f'{r.irr:.2%}' if r.irr else 'N/A'
    print(f'{name:<25} {irr_str:>8} ${r.pv_profits:>11,.0f} {r.profit_margin:>7.2%}')

worst_name, worst = scenario_result.worst_case()
irr_min, irr_max = scenario_result.irr_range()
print(f'\nWorst case: {worst_name} (IRR: {worst.irr:.2%})')
print(f'IRR range: {irr_min:.2%} — {irr_max:.2%}')

## 8. Cash Flow Visualisation

In [ ]:
years = np.arange(config.projection_months) / 12

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Polaris RE — Term Life YRT Pricing: Cash Flow Analysis', fontsize=14, y=1.02)

# Panel 1: Premiums and Claims (gross)
ax1 = axes[0, 0]
ax1.plot(years, gross.gross_premiums, label='Gross Premiums', color='steelblue')
ax1.plot(years, gross.death_claims, label='Death Claims', color='firebrick', linestyle='--')
ax1.set_title('Gross Premiums vs Claims')
ax1.set_xlabel('Year')
ax1.set_ylabel('Monthly Cash Flow ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Panel 2: Reserve balance
ax2 = axes[0, 1]
ax2.plot(years, gross.reserve_balance, color='darkorange')
ax2.set_title('Aggregate Reserve Balance')
ax2.set_xlabel('Year')
ax2.set_ylabel('Reserve ($)')
ax2.grid(True, alpha=0.3)

# Panel 3: Net cash flow
ax3 = axes[1, 0]
ax3.bar(years, net.net_cash_flow,
        color=['steelblue' if v >= 0 else 'firebrick' for v in net.net_cash_flow],
        alpha=0.7, width=1/12)
ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_title('Net Cash Flow (after YRT)')
ax3.set_xlabel('Year')
ax3.set_ylabel('Net Cash Flow ($)')
ax3.grid(True, alpha=0.3)

# Panel 4: Cumulative PV of profits
ax4 = axes[1, 1]
T = config.projection_months
v = (1.10) ** (-1/12)
discount_factors = v ** np.arange(1, T + 1)
cumulative_pv = np.cumsum(net.net_cash_flow * discount_factors)
ax4.plot(years, cumulative_pv, color='darkgreen')
ax4.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax4.fill_between(years, cumulative_pv, 0,
                 where=(cumulative_pv >= 0), color='lightgreen', alpha=0.4, label='Positive')
ax4.fill_between(years, cumulative_pv, 0,
                 where=(cumulative_pv < 0), color='lightcoral', alpha=0.4, label='Negative')
ax4.set_title('Cumulative PV of Profits @ 10%')
ax4.set_xlabel('Year')
ax4.set_ylabel('Cumulative PV ($)')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/output/term_life_yrt_cashflows.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to notebooks/output/')